# ELG Example Notebook

이 노트북은 `hypoevolve.elg` 패키지의 공개 API를 **설명 → 코드** 순서로 하나씩 보여준다.
각 셀은 독립적으로 읽기 쉽게 구성했고, 구조체 / 렌더링 / mutation helper / codec / normalize / metrics 순서로 정리했다.


## 1. 전체 공개 API import

먼저 이후 예시들에서 사용할 `hypoevolve.elg` 공개 API를 한 번에 import한다.


In [ ]:
from hypoevolve.elg import (
    AtomicNode,
    Hypothesis,
    LogicalNode,
    LogicalOp,
    Path,
    RelationNode,
    RelationType,
    count_atomics,
    count_logicals,
    count_nodes,
    count_relations,
    fingerprint,
    get_node_at_path,
    hypothesis_from_dict,
    hypothesis_from_json,
    hypothesis_to_json,
    iter_paths,
    mutate_append_child,
    mutate_logical_operator,
    mutate_relation_type,
    mutate_remove_child,
    mutate_replace_child,
    mutate_replace_subtree,
    mutate_unwrap_not,
    mutate_wrap_not,
    node_from_dict,
    node_to_dict,
    normalize_hypothesis,
    normalize_node,
    render_pretty,
    render_tree,
    replace_at_path,
    tree_depth,
)
import random


## 2. Enum 타입들

`LogicalOp`, `RelationType`은 현재 ELG의 닫힌 vocabulary를 나타낸다.
`AtomicNode`는 별도 enum 대신 문자열 이름만 보존한다.


In [ ]:
print(list(LogicalOp))
print(list(RelationType))


## 3. AtomicNode 생성

`AtomicNode`는 더 이상 ELG 내부에서 분해하지 않는 opaque leaf proposition이다.

In [ ]:
primitive_atomic = AtomicNode("FUNDING_FEE > 0")
semantic_atomic = AtomicNode("시장 상태가 불안정하다")

print(primitive_atomic)
print(semantic_atomic)
print(primitive_atomic.kind)


## 4. LogicalNode 생성

`LogicalNode`는 `AND`, `OR`, `NOT` 같은 논리 연산자를 가진다.

In [ ]:
and_node = LogicalNode(LogicalOp.AND, [
    AtomicNode("FUNDING_FEE > 0"),
    AtomicNode("CLOSE > SMA_20"),
])
not_node = LogicalNode(LogicalOp.NOT, [AtomicNode("RETURN_5M > 0")])

print(and_node)
print(not_node)
print(and_node.kind, and_node.op)

## 5. RelationNode 생성

`RelationNode`는 조건과 결과를 연결하는 관계 노드다.

In [ ]:
relation = RelationNode(
    RelationType.IMPLIES,
    [and_node, AtomicNode("RETURN_5M > 0")],
)

print(relation)
print(relation.kind)
print("condition =", relation.condition)
print("target =", relation.target)

## 6. Hypothesis 생성

`Hypothesis`는 root node 하나를 감싸는 최상위 wrapper다.

In [ ]:
hypothesis = Hypothesis(root=relation)
print(hypothesis)

## 7. node_to_dict

`node_to_dict`는 단일 node를 dict로 직렬화한다.

In [ ]:
print(node_to_dict(and_node))
print(node_to_dict(relation))

## 8. hypothesis_to_json

`hypothesis_to_json`은 hypothesis 전체를 JSON 문자열로 만든다.

In [ ]:
payload_json = hypothesis_to_json(hypothesis)
print(payload_json)

## 9. node_from_dict

`node_from_dict`는 `kind`를 기준으로 atomic / logical / relation node를 복원한다.

In [ ]:
node_payload = {
    "kind": "logical",
    "name": "OR",
    "inputs": [
        {"kind": "atomic", "name": "A"},
        {"kind": "atomic", "name": "B"},
    ],
}
restored_node = node_from_dict(node_payload)
print(restored_node)
print(render_pretty(restored_node))


## 10. hypothesis_from_dict

dict payload를 다시 `Hypothesis` 객체로 복원할 수 있다.

In [ ]:
hypothesis_payload = hypothesis.to_dict()
restored_hypothesis = hypothesis_from_dict(hypothesis_payload)
print(restored_hypothesis)
print(render_pretty(restored_hypothesis))

## 11. hypothesis_from_json

JSON 문자열로부터 hypothesis를 복원한다.

In [ ]:
restored_from_json = hypothesis_from_json(payload_json)
print(restored_from_json)
print(render_pretty(restored_from_json))

## 12. render_pretty

`render_pretty`는 사람이 읽기 좋은 중첩 표현으로 ELG를 보여준다.

In [ ]:
print(render_pretty(hypothesis))

## 13. render_tree

`render_tree`는 ASCII tree 형태로 ELG 구조를 시각화한다.

In [ ]:
print(render_tree(hypothesis))

## 14. normalize_node

단일 node를 정규화한다. 예를 들어 중복 child 제거, child 정렬, double negation 제거 같은 처리를 한다.

In [ ]:
unnormalized_node = LogicalNode("AND", [
    AtomicNode("B"),
    AtomicNode("A"),
    AtomicNode("A"),
])
normalized_node = normalize_node(unnormalized_node)
print(render_pretty(unnormalized_node))
print('---')
print(render_pretty(normalized_node))

## 15. normalize_hypothesis

전체 hypothesis를 정규화한다.

In [ ]:
unnormalized_hypothesis = Hypothesis(
    root=LogicalNode("NOT", [LogicalNode("NOT", [AtomicNode("A")])])
)
normalized_hypothesis = normalize_hypothesis(unnormalized_hypothesis)
print(render_pretty(unnormalized_hypothesis))
print('---')
print(render_pretty(normalized_hypothesis))

## 16. 구조 메트릭: count_nodes, tree_depth, count_atomics, count_logicals, count_relations

ELG의 구조적 복잡도를 정량화하는 기본 함수들이다.

In [ ]:
print("count_nodes =", count_nodes(hypothesis))
print("tree_depth =", tree_depth(hypothesis))
print("count_atomics =", count_atomics(hypothesis))
print("count_logicals =", count_logicals(hypothesis))
print("count_relations =", count_relations(hypothesis))

## 17. fingerprint

`fingerprint`는 정규화된 구조를 기반으로 해시를 만든다. 구조적으로 같은 가설은 같은 fingerprint를 갖는다.

In [ ]:
h1 = Hypothesis(root=LogicalNode("AND", [AtomicNode("A"), AtomicNode("B")]))
h2 = Hypothesis(root=LogicalNode("AND", [AtomicNode("B"), AtomicNode("A")]))

print(fingerprint(h1))
print(fingerprint(h2))
print("same fingerprint:", fingerprint(h1) == fingerprint(h2))

## 18. Path 타입

`Path`는 트리 안의 특정 위치를 가리키는 tuple 기반 주소다.

- `()` = root
- `(0,)` = 첫 번째 child
- `(0, 1)` = root의 첫 child의 두 번째 child

In [ ]:
root_path: Path = ()
condition_path: Path = (0,)
second_condition_child_path: Path = (0, 1)

a = root_path, condition_path, second_condition_child_path
print(a)

## 19. get_node_at_path

주어진 path의 node를 가져온다.

In [ ]:
print(get_node_at_path(hypothesis, ()))
print(get_node_at_path(hypothesis, (0,)))
print(get_node_at_path(hypothesis, (0, 1)))

## 20. iter_paths

현재 hypothesis에서 접근 가능한 모든 path를 순회한다.

In [ ]:
print(iter_paths(hypothesis))

## 21. replace_at_path

지정한 path의 subtree를 새 node로 교체한다. 원본 hypothesis는 바뀌지 않는다.

In [ ]:
replaced = replace_at_path(hypothesis, (0, 1), AtomicNode("OPEN_INTEREST_CHANGE > 0"))
print(render_pretty(replaced))
print('--- original ---')
print(render_pretty(hypothesis))

## 22. mutate_replace_subtree

`replace_at_path`의 mutation-friendly wrapper로, subtree 전체를 새 구조로 바꾼다.

In [ ]:
subtree_mutated = mutate_replace_subtree(
    hypothesis,
    (0,),
    LogicalNode("OR", [AtomicNode("VOLATILITY_HIGH"), AtomicNode("FUNDING_FEE > 0")]),
)
print(render_pretty(subtree_mutated))

## 23. mutate_replace_child

논리 노드나 관계 노드의 특정 child만 교체한다.

In [ ]:
child_mutated = mutate_replace_child(hypothesis, (0,), 1, AtomicNode("VOLUME > AVG_VOLUME"))
print(render_pretty(child_mutated))

## 24. mutate_logical_operator

`AND -> OR`, `OR -> AND` 같은 logical operator 변경을 수행한다.

In [ ]:
logical_mutated = mutate_logical_operator(hypothesis, (0,), "OR")
print(render_pretty(logical_mutated))

## 25. mutate_relation_type

relation 종류를 바꾼다.

In [ ]:
relation_mutated = mutate_relation_type(hypothesis, (), "SUPPORT")
print(render_pretty(relation_mutated))

## 26. mutate_wrap_not

지정한 path의 node를 `NOT(node)`로 감싼다.

In [ ]:
wrapped_not = mutate_wrap_not(hypothesis, (1,))
print(render_pretty(wrapped_not))

## 27. mutate_unwrap_not

`NOT(A)`를 다시 `A`로 푼다.

In [ ]:
unwrapped = mutate_unwrap_not(wrapped_not, (1,))
print(render_pretty(unwrapped))

## 28. mutate_append_child

`AND/OR` 노드에 child를 하나 더 추가한다.

In [ ]:
appended = mutate_append_child(hypothesis, (0,), AtomicNode("VOLUME > AVG_VOLUME"))
print(render_pretty(appended))

## 29. mutate_remove_child

`AND/OR` 노드에서 특정 child를 제거한다. 최소 arity를 깨면 예외가 난다.

In [ ]:
removed = mutate_remove_child(appended, (0,), 1)
print(render_pretty(removed))

## 30. Path enumeration helpers

`iter_paths(...)`와 `get_node_at_path(...)`를 사용하면 현재 hypothesis의 각 노드를 안정적으로 가리킬 수 있다.


In [ ]:
path_demo = Hypothesis(
    root=LogicalNode("AND", [AtomicNode("A"), AtomicNode("B")])
)
paths = iter_paths(path_demo)
print(paths)
print(get_node_at_path(path_demo, (1,)))


## 31. mutate_replace_subtree

현재 hypothesis의 특정 path에 있는 서브트리를 다른 노드로 교체할 수 있다.


In [ ]:
replace_demo = Hypothesis(
    root=LogicalNode("AND", [AtomicNode("A"), AtomicNode("B")])
)
replaced = mutate_replace_subtree(replace_demo, (1,), AtomicNode("D"))
print(render_pretty(replace_demo))
print("---")
print(render_pretty(replaced))


## 32. mutate_wrap_not / mutate_unwrap_not

임의의 노드를 `NOT`으로 감쌌다가 다시 벗기는 예시다.


In [ ]:
not_demo = Hypothesis(root=LogicalNode("AND", [AtomicNode("A"), AtomicNode("B")]))
wrapped = mutate_wrap_not(not_demo, (1,))
print(render_pretty(wrapped))
print("---")
unwrapped = mutate_unwrap_not(wrapped, (1,))
print(render_pretty(unwrapped))


## 33. 전체 흐름 예시

마지막으로 ELG를 한 번에 생성하고, 렌더링하고, mutate하고, serialize하고, normalize/fingerprint를 계산하는 전체 흐름을 보여준다.

In [ ]:
base = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode("AND", [
                AtomicNode("FUNDING_FEE > 0"),
                AtomicNode("CLOSE > SMA_20"),
            ]),
            AtomicNode("RETURN_5M > 0"),
        ],
    ),
)

print('PRETTY')
print(render_pretty(base))
print('---')
print('TREE')
print(render_tree(base))
print('---')
mutated = mutate_append_child(base, (0,), AtomicNode("VOLATILITY_HIGH"))
print(render_pretty(mutated))
print('---')
json_payload = hypothesis_to_json(mutated)
print(json_payload)
print('---')
restored = hypothesis_from_json(json_payload)
print('fingerprint =', fingerprint(restored))
print('normalized =')
print(render_pretty(normalize_hypothesis(restored)))
print('metrics =', count_nodes(restored), tree_depth(restored), count_atomics(restored), count_logicals(restored), count_relations(restored))


## 34. LLMClient with local vLLM

이 예시는 `hypoevolve.llm.LLMClient`를 **로컬 vLLM OpenAI-compatible endpoint** 와 함께 사용하는 방법을 보여준다.

- backend: local vLLM (OpenAI-compatible)
- model: `DeepSeek-R1-Distill-Qwen-14B`

> 보안상 **실제 API 키를 노트북에 직접 저장하지 말고**, 환경변수로 주입하는 방식을 사용한다.

In [ ]:
import os
from hypoevolve import LLMClient, LLMConfig

# 로컬 vLLM 엔드포인트는 별도 OPENAI_API_KEY 없이 사용 가능
# 또는 이 셀에서 임시로 설정하기 (노트북 저장 전 제거 권장)
# 필요하면 api_key="EMPTY" 같은 더미 값을 명시해도 된다
API_KEY = "EMPTY"

llm_config = LLMConfig(
    model="DeepSeek-R1-Distill-Qwen-14B",
    api_base="http://127.0.0.1:8000/v1",
    api_key=API_KEY,
    temperature=0.2,
    max_tokens=1000,
    timeout=60,
    retries=2,
    retry_delay=1.0,
)

client = LLMClient(llm_config)
print(llm_config)

## 35. generate_text 예시

`generate_text()`는 system prompt와 user prompt를 받아 일반 텍스트 응답을 반환한다.

In [ ]:
system_prompt = "You are a concise research assistant."
user_prompt = "Summarize why structured hypothesis mutation can be more stable than raw natural-language mutation in 3 bullets."

# 실제 호출 예시
response = client.generate_text(system_prompt, user_prompt)
print(response)

print("client.generate_text(system_prompt, user_prompt)")

## 36. generate_json 예시

`generate_json()`는 LLM에게 JSON만 반환하도록 지시하고, 그 응답을 Python `dict`로 파싱한다.

실전에서는 출력 계약을 분명히 적는 것이 중요하다.

In [ ]:
system_prompt = "Return only valid JSON."
user_prompt = """
Return a JSON object with keys:
- score: float between 0 and 1
- rationale: short string
for the hypothesis: 'If funding fee is positive then short-term returns are positive.'
"""

# 실제 호출 예시
payload = client.generate_json(system_prompt, user_prompt)
print(payload)

print("client.generate_json(system_prompt, user_prompt)")

In [ ]:
payload

## 40. 실제 LLM parser 호출 예시

이 셀은 `LLMClient`와 `llm_parse_hypothesis()`를 실제로 연결해서, 자연어 가설을 ELG로 변환하는 예시다.

- backend: local vLLM (OpenAI-compatible)
- model: `DeepSeek-R1-Distill-Qwen-14B`

> 로컬 vLLM endpoint면 API key 없이도 동작한다.

In [ ]:
from hypoevolve import LLMClient, LLMConfig
from hypoevolve.parser import llm_parse_hypothesis
from hypoevolve.prompts import load_prompt
from hypoevolve.elg import render_pretty, render_tree

API_KEY = "EMPTY"

llm_config = LLMConfig(
    model="DeepSeek-R1-Distill-Qwen-14B",
    api_base="http://127.0.0.1:8000/v1",
    api_key=API_KEY,  # 로컬 vLLM이면 EMPTY 같은 더미 값이면 충분
    temperature=0.2,
    max_tokens=1200,
    timeout=60,
    retries=2,
    retry_delay=1.0,
)

client = LLMClient(llm_config)


## 41. 자연어 가설을 ELG로 파싱해보기

아래 셀은 실제로 자연어 가설 문자열을 LLM parser로 보내고, 반환된 ELG를 pretty/tree 형태로 확인하는 예시다.

In [ ]:
hypothesis_text = "The statistical dependency suggests that sharp downward accelerations in BTCUSDT price, indicated by the NewLow signal derived from the normalized 10-day close momentum, are systematically followed by significant jumps in the transformed high ETHUSDT price series. This relationship likely captures a 'squeeze-and-release' dynamic where extreme negative momentum (a new low) creates oversold conditions, triggering a subsequent short-covering rally or reversal spike that manifests as a jump in the exponential z-score of recent highs. The hypothesis predicts that markets exhibiting this pattern have an asymmetric volatility response to new lows, where the initial sharp sell-off reliably forces a transient, high-magnitude recovery spike within the same period."

# 실제 호출
parsed_hypothesis = llm_parse_hypothesis(hypothesis_text, llm=client, retries=2)

print("=== PRETTY ===")
print(render_pretty(parsed_hypothesis))
print()
print("=== TREE ===")
print(render_tree(parsed_hypothesis))
print()
print("=== RAW DICT ===")
print(parsed_hypothesis.to_dict())

In [ ]:
try:
    parsed_hypothesis = llm_parse_hypothesis(hypothesis_text, llm=client, retries=2)
except Exception as e:
    print(type(e).__name__, e)
    if hasattr(e, "errors"):
        print("---- detailed errors ----")
        for item in e.errors:
            print(item)

In [ ]:
from hypoevolve.parser import PARSER_SYSTEM_PROMPT, PARSER_RETRY_PROMPT

system_prompt = PARSER_SYSTEM_PROMPT
user_prompt = f"Convert this natural-language hypothesis into ELG JSON root node:\n\n{hypothesis_text}"

raw = client.generate_text(system_prompt, user_prompt)
print(raw)

payload = client.generate_json(system_prompt, user_prompt)
print(payload)

## 42. semantic ELG를 measurable ELG로 변환하기

이 셀은 이미 만들어진 semantic ELG hypothesis를 받아서, `llm_make_hypothesis_measurable()`를 통해 더 measurable 한 ELG로 바꾸는 예시다.

핵심 아이디어:
- relation / logical 구조는 최대한 유지
- atomic proposition을 더 계산 가능한 형태로 구체화

In [ ]:
from hypoevolve import llm_make_hypothesis_measurable
from hypoevolve.elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, render_pretty, render_tree

semantic_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            AtomicNode(
                "sharp downward accelerations in BTCUSDT price indicated by NewLow signal from normalized close momentum"
            ),
            AtomicNode(
                "significant jumps in high ETHUSDT price series"
            ),
        ],
    )
)

print("=== SEMANTIC ===")
print(render_pretty(semantic_hypothesis))

## 43. measurable 변환 실제 호출 예시

이 셀은 실제로 LLM을 호출해 measurable ELG를 생성하는 예시다.

In [ ]:
measurable_hypothesis = llm_make_hypothesis_measurable(
    semantic_hypothesis,
    llm=client,
    retries=2,
)

print("=== MEASURABLE / PRETTY ===")
print(render_pretty(measurable_hypothesis))
print()
print("=== MEASURABLE / TREE ===")
print(render_tree(measurable_hypothesis))
print()
print("=== MEASURABLE / RAW DICT ===")
print(measurable_hypothesis.to_dict())

## 44. ELG를 자연어 문장으로 바꾸기

이 셀은 이미 구성된 ELG hypothesis를 받아서, `llm_hypothesis_to_natural_language()`로 사람이 읽기 좋은 자연어 문장으로 바꾸는 예시다.

In [ ]:
from hypoevolve import llm_hypothesis_to_natural_language

natural_language_text = llm_hypothesis_to_natural_language(
    measurable_hypothesis,
    llm=client,
    retries=2,
)

print(natural_language_text)

## 45. LocalSubprocessExecutor 사용 예시

이 셀은 `LocalSubprocessExecutor`를 사용해서 LLM이 생성했다고 가정한 Python 코드를 임시 작업 디렉토리에서 실행하고, 결과를 회수하는 예시다.

In [ ]:
from hypoevolve import LocalSubprocessExecutor

executor = LocalSubprocessExecutor(timeout=10, cleanup=False)

result = executor.execute(
    """
print('hello from generated code')
value = 21 * 2
print(f'value={value}')
"""
)

print("stdout:")
print(result.stdout)
print("stderr:")
print(result.stderr)
print("exit_code:", result.exit_code)
print("timed_out:", result.timed_out)
print("duration_sec:", result.duration_sec)
print("work_dir:", result.work_dir)

## 46. parquet 기반 DatasetSchema 예시

이 셀은 여러 parquet 파일을 entity별로 묶어서 `DatasetSchema`를 만드는 예시다.

가정:
- `BTCUSDT.parquet`
- `ETHUSDT.parquet`
같이 자산별 parquet 파일이 따로 있음

In [ ]:
from hypoevolve import ColumnSpec, DataFile, DatasetSchema, DatasetAccessor
from hypoevolve.dataset import IndexSpec

schema = DatasetSchema(
    files=[
        DataFile(entity="BTCUSDT", path="data/BTCUSDT.parquet"),
        DataFile(entity="ETHUSDT", path="data/ETHUSDT.parquet"),
    ],
    index=IndexSpec(name="close_time", dtype="datetime64[us]"),
    columns=[
        ColumnSpec(name="open", description="open price"),
        ColumnSpec(name="high", description="high price"),
        ColumnSpec(name="low", description="low price"),
        ColumnSpec(name="close", description="close price"),
        ColumnSpec(name="volume", description="traded volume"),
        ColumnSpec(name="funding_fee", description="funding fee at each timestamp"),
    ],
    description="Per-asset OHLCV parquet dataset",
)

accessor = DatasetAccessor(schema)
print(accessor.summary())


## 47. YAML 기반 DatasetSchema 로딩 예시

실전에서는 dataset schema를 코드에 직접 쓰기보다 YAML 파일로 관리하는 편이 더 편하다.

In [ ]:
from hypoevolve import load_dataset_schema
from hypoevolve import DatasetAccessor

# 예시 파일 형태:
# dataset.yaml
# description: Per-asset OHLCV parquet dataset
# index:
#   name: close_time
#   dtype: datetime64[us]
# files:
#   -
#     entity: BTCUSDT
#     path: data/BTCUSDT.parquet
#   -
#     entity: ETHUSDT
#     path: data/ETHUSDT.parquet
# columns:
#   -
#     name: close
#     description: close price
#   -
#     name: funding_fee
#     description: funding fee at each timestamp

loaded_schema = load_dataset_schema("dataset.yaml")
accessor = DatasetAccessor(loaded_schema)
print(accessor.summary())

print("load_dataset_schema('dataset.yaml')")

## 48. DatasetAccessor 사용 예시

이 셀은 accessor가 어떤 helper를 제공하는지 보여준다.

In [ ]:
print(accessor.entities())
print(accessor.column_names())
print(accessor.column_descriptions())

# 실제 parquet 파일이 있을 때:
rows = accessor.head("BTCUSDT", n=5)
print(rows)

## 49. prompt 변수 치환 helper 예시

이 셀은 `load_prompt`, `render_prompt`, `load_and_render_prompt`를 사용해서 evaluator용 user prompt 템플릿에 실제 변수들을 채워 넣는 예시다.

In [ ]:
from hypoevolve.prompts import load_prompt, render_prompt, load_and_render_prompt
from hypoevolve.elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, render_pretty

example_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode("AND", [
                AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_10D == True"),
                AtomicNode("BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D < -2.0"),
            ]),
            AtomicNode("ETHUSDT_TRANSFORMED_HIGH_JUMP_10D > 1.5"),
        ],
    )
)

variables = {
    "HYPOTHESIS_PRETTY": render_pretty(example_hypothesis),
    "HYPOTHESIS_JSON": example_hypothesis.to_dict(),
    "DATASET_DESCRIPTION": "Per-asset OHLCV parquet dataset",
    "TIME_COLUMN": "timestamp",
    "ENTITIES": "BTCUSDT, ETHUSDT",
    "COLUMN_SPECS": "- close: close price - high: high price - funding_fee: funding fee",
    "DATASET_ACCESSOR_DOC": "accessor.load_dataframe(entity), accessor.head(entity, n=5), accessor.summary()",
    "DATASET_SAMPLES": "BTCUSDT.head(5), ETHUSDT.head(5)",
}


## 50. 템플릿 파일 로드 후 직접 렌더링하기

먼저 템플릿을 직접 로드하고 `render_prompt()`로 치환하는 방식이다.

In [ ]:
user_template = load_prompt("evaluator", "user.md")
rendered_user_prompt = render_prompt(user_template, variables)
print(rendered_user_prompt)

## 51. load_and_render_prompt 한 번에 쓰기

파일 로드와 변수 치환을 한 번에 수행하는 방식이다.

In [ ]:
rendered_user_prompt_2 = load_and_render_prompt(
    "evaluator",
    "user.md",
    variables=variables,
)
print(rendered_user_prompt_2)

## 52. evaluator code generation inference 예시

이 셀은 `prompts/evaluator/system.md`와 `prompts/evaluator/user.md`를 로드하고,
실제 변수들을 채운 뒤 `LLMClient.generate_text()`를 호출해서 **evaluator용 Python 코드 초안**을 받아보는 예시다.

In [ ]:
from hypoevolve import load_dataset_schema
from hypoevolve.dataset import DatasetAccessor
from hypoevolve.prompts import load_prompt, load_and_render_prompt
from hypoevolve.elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, render_pretty
import json

schema = load_dataset_schema("dataset.yaml")
accessor = DatasetAccessor(schema)
accessor_doc = """
DatasetAccessor methods:
- accessor.entities() -> list[str]
- accessor.column_names() -> list[str]
- accessor.column_descriptions() -> dict[str, str]
- accessor.load_dataframe(entity) -> pandas.DataFrame
- accessor.load_all_dataframes() -> dict[str, pandas.DataFrame]
- accessor.head(entity, n=5) -> pandas.DataFrame
- accessor.summary() -> dict
""".strip()

measurable_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode(
                "AND",
                [
                    AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_10D@t == True"),
                    AtomicNode("BTCUSDT_NORMALIZED_CLOSE_MOMENTUM_10D@t < -2.0"),
                ],
            ),
            AtomicNode("ETHUSDT_TRANSFORMED_HIGH_JUMP_10D@t+1 > 1.5"),
        ],
    )
)

sample_data = {
    entity: accessor.head(entity, 3).to_dict(orient="records")
    for entity in accessor.entities()[:3]
}

variables = {
    "HYPOTHESIS_PRETTY": render_pretty(measurable_hypothesis),
    "DATASET_DESCRIPTION": schema.description or "",
    "TIME_COLUMN": schema.index.name,
    "ENTITIES": ", ".join(accessor.entities()),
    "COLUMN_SPECS": "\n".join(
        f"- {column.name}: {column.description or ''}".rstrip()
        for column in schema.columns
    ),
    "DATASET_ACCESSOR_DOC": accessor_doc,
    "DATASET_SAMPLES": json.dumps(sample_data, ensure_ascii=False, indent=2),
}


## 53. 실제 evaluator code generation 호출

이 셀은 위에서 만든 prompt를 그대로 `LLMClient`에 넣고, 반환된 Python 코드를 출력하는 예시다.

In [ ]:
print(variables["HYPOTHESIS_PRETTY"])

In [ ]:
generated_code = client.generate_text(system_prompt, user_prompt)
print(generated_code)

## 54. LLMEvaluator 사용 예시


In [ ]:
from hypoevolve import LLMClient, LLMEvaluator, load_config, load_dataset_schema
from hypoevolve.elg import AtomicNode, Hypothesis, LogicalNode, RelationNode

config = load_config("hypoevolve.yaml")
client = LLMClient(config.llm)
schema = load_dataset_schema("dataset.yaml")

hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode(
                "AND",
                [
                    AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_10@t == True"),
                    AtomicNode("BTCUSDT_ZSCORE_CLOSE_MOMENTUM_10@t < -2.0"),
                ],
            ),
            AtomicNode("DOGEUSDT_ZSCORE_HIGH_JUMP_10@t+1 > 1.5"),
        ],
    )
)

evaluator = LLMEvaluator(
    llm_client=client,
    dataset_schema=schema,
    dataset_schema_path="dataset.yaml",
)

metrics = evaluator.evaluate(hypothesis)
metrics

## 55. mutation steering 사용 예시

이 예시는 현재 mutation steering이 **candidate index를 고르는 방식이 아니라**, parent measurable ELG와 현재 metric 문맥을 바탕으로 **새 child ELG를 직접 생성하는 방식**으로 동작하는 모습을 보여준다.

출력은 full child ELG이지만, 프롬프트는 여전히 `replace_atomic`, `change_logical_operator`, `append_child` 같은 **local mutation 스타일**을 따르도록 유도한다. 따라서 결과를 읽을 때는 `mutation_summary`를 통해 parent 대비 어떤 국소 변경이 적용되었는지 함께 확인하면 된다.


In [ ]:
from hypoevolve import LLMClient, load_config, steer_mutation
from hypoevolve.parser import ParseError
from hypoevolve.elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, render_pretty

config = load_config("hypoevolve.yaml")
client = LLMClient(config.llm)

parent_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode(
                "AND",
                [
                    AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_W{MOM_WINDOW}@t == True"),
                    AtomicNode("BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W{RET_WINDOW}@t < {NEG_Z_THRESHOLD}"),
                ],
            ),
            AtomicNode("DOGEUSDT_ZSCORE_HIGH_JUMP_W{TARGET_WINDOW}@t+{HORIZON} > {POS_Z_THRESHOLD}"),
        ],
    )
)

current_metrics = {
    "combined_score": -0.00045,
    "precision": 0.0216,
    "baseline": 0.0682,
    "coverage": 0.0098,
    "uplift": -0.0466,
    "support_count": 4115,
    "total_count": 420768,
    "rationale": "Condition occurs 4115/420768 times. When condition is true, target occurs 2.2% vs baseline 6.8%."
}

recent_history = [
    {
        "mutation_summary": "Applied a replace_atomic_feature mutation on the target side by changing DOGEUSDT_ZSCORE_HIGH_JUMP_W{TARGET_WINDOW}@t+{HORIZON} > {POS_Z_THRESHOLD} to DOGEUSDT_ZSCORE_CLOSE_RETURN_W{TARGET_WINDOW}@t+{HORIZON} > {POS_Z_THRESHOLD}.",
        "result_hypothesis": "IMPLIES(AND(BTCUSDT_NEW_LOW_SIGNAL_W{MOM_WINDOW}@t == True, BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W{RET_WINDOW}@t < {NEG_Z_THRESHOLD}), DOGEUSDT_ZSCORE_CLOSE_RETURN_W{TARGET_WINDOW}@t+{HORIZON} > {POS_Z_THRESHOLD})",
        "note": "A target feature change was tried recently."
    },
    {
        "mutation_summary": "Applied a replace_atomic_direction mutation on the target side by changing DOGEUSDT_ZSCORE_HIGH_JUMP_W{TARGET_WINDOW}@t+{HORIZON} > {POS_Z_THRESHOLD} to DOGEUSDT_ZSCORE_HIGH_JUMP_W{TARGET_WINDOW}@t+{HORIZON} < {NEG_Z_THRESHOLD}.",
        "result_hypothesis": "IMPLIES(AND(BTCUSDT_NEW_LOW_SIGNAL_W{MOM_WINDOW}@t == True, BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W{RET_WINDOW}@t < {NEG_Z_THRESHOLD}), DOGEUSDT_ZSCORE_HIGH_JUMP_W{TARGET_WINDOW}@t+{HORIZON} < {NEG_Z_THRESHOLD})",
        "note": "A target direction flip was also tried recently."
    },
]

try:
    decision = steer_mutation(
        parent_hypothesis=parent_hypothesis,
        current_metrics=current_metrics,
        llm=client,
        recent_history=recent_history,
        top_hypotheses=[],
        retries=2,
    )
except ParseError as exc:
    print("steering failed:", exc)
    if exc.errors:
        print("attempt details:")
        for item in exc.errors:
            print("-", item)
    raise

print("score_reason:", decision.score_reason)
print("domain_reason:", decision.domain_reason)
print("mutation_summary:", decision.mutation_summary)
print("child hypothesis:")
print(render_pretty(decision.child_hypothesis))


## 56. random exploration mutation steering 사용 예시

이 예시는 score를 직접 최적화하지 않고, 최근 mutation history와 다른 방향의 local mutation을 생성하는 exploration용 steering 프롬프트 사용 예시다.


In [ ]:
from hypoevolve import LLMClient, load_config
from hypoevolve.helper import build_steering_prompt_variables
from hypoevolve.prompts import load_and_render_prompt, load_prompt
from hypoevolve.elg import AtomicNode, Hypothesis, LogicalNode, RelationNode, hypothesis_from_dict, render_pretty
import json

config = load_config("hypoevolve.yaml")
client = LLMClient(config.llm)

parent_hypothesis = Hypothesis(
    root=RelationNode(
        "IMPLIES",
        [
            LogicalNode(
                "AND",
                [
                    AtomicNode("BTCUSDT_NEW_LOW_SIGNAL_W{MOM_WINDOW}@t == True"),
                    AtomicNode("BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W{RET_WINDOW}@t < {NEG_Z_THRESHOLD}"),
                ],
            ),
            AtomicNode("DOGEUSDT_ZSCORE_HIGH_JUMP_W{TARGET_WINDOW}@t+{HORIZON} > {POS_Z_THRESHOLD}"),
        ],
    )
)

recent_history = [
    {
        "mutation_summary": "Applied a replace_atomic_feature mutation by changing the target observable from HIGH_JUMP to CLOSE_RETURN while preserving parameter slots.",
        "result_hypothesis": "IMPLIES(AND(BTCUSDT_NEW_LOW_SIGNAL_W{MOM_WINDOW}@t == True, BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W{RET_WINDOW}@t < {NEG_Z_THRESHOLD}), DOGEUSDT_ZSCORE_CLOSE_RETURN_W{TARGET_WINDOW}@t+{HORIZON} > {POS_Z_THRESHOLD})",
        "note": "A target feature variation was explored recently."
    },
    {
        "mutation_summary": "Applied a replace_atomic_direction mutation by flipping the target polarity while preserving parameter slots.",
        "result_hypothesis": "IMPLIES(AND(BTCUSDT_NEW_LOW_SIGNAL_W{MOM_WINDOW}@t == True, BTCUSDT_ZSCORE_CLOSE_MOMENTUM_W{RET_WINDOW}@t < {NEG_Z_THRESHOLD}), DOGEUSDT_ZSCORE_HIGH_JUMP_W{TARGET_WINDOW}@t+{HORIZON} < {NEG_Z_THRESHOLD})",
        "note": "A target direction variation was also explored recently."
    },
]

variables = build_steering_prompt_variables(
    parent_hypothesis=parent_hypothesis,
    current_metrics={},
    recent_history=recent_history,
    top_hypotheses=[],
)

system_prompt = load_prompt("steering-random", "system.md")
user_prompt = load_and_render_prompt("steering-random", "user.md", variables=variables)

payload = client.generate_json(system_prompt, user_prompt)
child_hypothesis = hypothesis_from_dict({"root": payload["child_hypothesis"]})

print("mutation_summary:", payload["mutation_summary"])
print("child_hypothesis instance:")
print(render_pretty(child_hypothesis))
